# Lecture 00 — What is the web, really?

> *"You can't crawl what you don't understand. Most people who have trouble writing crawlers have trouble because they're guessing about what the browser is doing on their behalf."*

This lecture has nothing to do with crawling, and that's the point. Before you can extract data from the web you need a working mental model of what *the web* is. We will rebuild that model from scratch.

## What you'll be able to do after this lecture

- Describe what happens, step by step, when you type a URL into your browser and press Enter.
- Read a raw HTTP request and response and explain every line.
- Explain the difference between a *server-rendered* and a *client-rendered* page — and why that distinction will eat half of your future debugging time.
- Use your browser's DevTools to inspect any of these things on a live site.

## Prerequisites

- Python 3.10+ installed
- `pip install httpx beautifulsoup4 lxml`
- A modern browser (Chromium-based or Firefox is fine)


## 1. The web is just two computers talking

When you type `https://example.com` into your browser, you are starting a conversation. Your computer (the **client**) asks another computer (the **server**) for something, and the server replies. That's it. Everything else — HTML, CSS, JavaScript, login forms, infinite scroll, captchas — is detail layered on top of that two-line conversation.

The conversation has rules. The rules have a name: **HTTP** (HyperText Transfer Protocol). When the conversation is encrypted (which it almost always is in 2026), it's called **HTTPS**.

Here is what the conversation actually looks like, in spirit:

```
CLIENT: Hi, I'd like to GET the document at /index.html, please.
SERVER: Sure, here it is. It's HTML, it's 12KB, and here are some cookies for next time.
```

That's the whole web. The rest of this lecture is just zooming in on the parts of that exchange that matter for crawling.


## 2. What happens when you type a URL

Let's slow down the moment between **press Enter** and **see the page**:

1. **DNS lookup.** `example.com` is a name, not an address. Your computer asks a DNS resolver: *"Where does `example.com` live?"* It comes back with an IP address like `93.184.216.34`. (For a crawler, this matters because DNS lookups take time and can fail. Reusing connections matters.)
2. **TCP connection.** Your computer opens a network connection to that IP address on port 443 (HTTPS) or 80 (HTTP).
3. **TLS handshake.** For HTTPS, the two sides agree on encryption. They check a certificate so the server can prove it really is `example.com` and not someone pretending to be it.
4. **HTTP request.** Your client sends a request: a method (`GET`), a path (`/index.html`), some headers (User-Agent, Accept, Cookie, ...), and possibly a body.
5. **HTTP response.** The server replies: a status code (`200 OK`, `404 Not Found`, ...), some headers (Content-Type, Set-Cookie, ...), and a body (the HTML).
6. **Browser parses HTML.** It finds `<link>`, `<script>`, `<img>` tags and fires off *more* requests for those, recursively. A typical news article today triggers 50–200 sub-requests.
7. **Browser runs JavaScript.** The JS may rewrite the page, fetch more data via XHR/`fetch`, and only *then* show you the content you came to read.

A crawler is a program that pretends to be steps 4–6 of this sequence. The interesting question, lecture by lecture, is *how much* of step 6 and 7 do we have to imitate?


## 3. The HTTP request, dissected

Here is a real HTTP request, the kind your browser sends every time you click a link. We'll send one ourselves in a moment using Python.

```http
GET /index.html HTTP/1.1
Host: example.com
User-Agent: Mozilla/5.0 (X11; Linux x86_64) ... Firefox/127.0
Accept: text/html,application/xhtml+xml,application/xml;q=0.9
Accept-Language: en-US,en;q=0.5
Accept-Encoding: gzip, deflate, br
Cookie: sessionid=abc123; theme=dark
Connection: keep-alive
```

Five things to notice, because each one will come back to bite you later:

- **Method (`GET`).** Other common methods: `POST` (submit a form, create something), `PUT`/`PATCH` (update), `DELETE`. Crawlers are 99% `GET`.
- **Path (`/index.html`).** Just the path, not the full URL. The host goes in the `Host` header.
- **Headers.** Free-form key:value lines that carry metadata. Servers use them to decide what to send you. Many anti-bot systems make their decision based purely on the *shape* of your headers.
- **`User-Agent`.** The string that says *what kind of client* you are. Default Python `urllib.request` sends `Python-urllib/3.11`, which is a "please block me" sign on many sites. We'll fix this.
- **`Cookie`.** Small bits of state the server gave you previously. Sessions, login state, A/B test buckets, all live here.


## 4. The HTTP response, dissected

```http
HTTP/1.1 200 OK
Content-Type: text/html; charset=utf-8
Content-Length: 1256
Set-Cookie: sessionid=abc123; HttpOnly
Cache-Control: max-age=600

<!DOCTYPE html>
<html>
<head><title>Example</title></head>
<body><h1>Hello</h1></body>
</html>
```

The first line is the **status line**. The number is the **status code**, and you must learn to read these like a doctor reads a chart:

| Range | Meaning            | Examples you'll meet                        |
|------:|--------------------|---------------------------------------------|
| 1xx   | Informational      | (rare; ignore)                              |
| 2xx   | Success            | `200 OK`, `204 No Content`                  |
| 3xx   | Redirect           | `301 Moved Permanently`, `302 Found`        |
| 4xx   | Client did wrong   | `403 Forbidden`, `404 Not Found`, `429 Too Many Requests` |
| 5xx   | Server did wrong   | `500 Internal Server Error`, `503 Service Unavailable` |

**Two response codes a crawler must handle gracefully:**
- `429 Too Many Requests` — you are being rate-limited. Slow down.
- `503 Service Unavailable` — server is overloaded or playing dead at you. Back off and retry.

The headers carry more metadata. The body is what you actually came for.


## 5. Send your first HTTP request from Python

We'll use `httpx`, a modern Python HTTP library. (`requests` is the older sibling and works almost identically. We'll mention both throughout.)


In [ ]:
import httpx

response = httpx.get("https://example.com")

print("Status:", response.status_code)
print("Type:  ", response.headers.get("content-type"))
print("Bytes: ", len(response.content))
print()
print("First 300 chars of body:")
print(response.text[:300])


Inspect every part of the request and response. Run the cell below — every dot here is a thing to know.

In [ ]:
# What did WE send?
print("=== REQUEST ===")
print(response.request.method, response.request.url)
for k, v in response.request.headers.items():
    print(f"  {k}: {v}")

print()
print("=== RESPONSE ===")
print("Status:", response.status_code, response.reason_phrase)
for k, v in response.headers.items():
    print(f"  {k}: {v}")


Notice: even though you didn't set any headers, `httpx` sent a few sensible defaults (`Host`, `User-Agent`, `Accept`, `Accept-Encoding`). This is friendlier than `urllib.request` but still identifiable as a Python script. We'll discuss when (and whether) to disguise this in lecture 08.

## 6. HTML, briefly

The body of most responses you'll care about is **HTML**. HTML is a tree.

```html
<html>
  <head>
    <title>Example</title>
  </head>
  <body>
    <h1>Hello</h1>
    <p class="intro">A paragraph.</p>
    <ul>
      <li>One</li>
      <li>Two</li>
    </ul>
  </body>
</html>
```

Two ideas:

- **Tags** are the boxes (`<h1>...</h1>`).
- **Attributes** are the labels on the boxes (`class="intro"`, `id="main"`, `href="..."`).

When a browser receives this HTML it builds an in-memory tree called the **DOM** (Document Object Model). Every `<tag>` becomes a node. Every text fragment becomes a leaf. CSS styles the nodes. JavaScript reads and modifies them.

For a crawler, the DOM is the data structure you query. You'll spend lecture 03 learning how to navigate it with BeautifulSoup.

In [ ]:
from bs4 import BeautifulSoup

html = response.text
soup = BeautifulSoup(html, "lxml")

# title of the page
print("Title:", soup.title.string)

# all paragraph text
print("Paragraphs:")
for p in soup.find_all("p"):
    print(" ", p.get_text(strip=True))

# all links
print("Links:")
for a in soup.find_all("a"):
    print(" ", a.get("href"), "->", a.get_text(strip=True))


## 7. The fork in the road: server-rendered vs client-rendered

This is the single most important distinction in modern web crawling. Pay attention.

**Server-rendered (SSR)** means: the server takes data from a database, plugs it into an HTML template, and sends you the finished HTML. When you `GET` the page, the response body already contains the content you care about. Older sites, blogs, news outlets, e-commerce listings — many are still SSR. `httpx.get(...)` + BeautifulSoup is enough.

**Client-rendered (CSR)** means: the server sends you a tiny HTML skeleton plus a big JavaScript bundle. The bundle runs in your browser, fetches data from a JSON API, and *builds* the page in your browser's memory. When you `GET` the page with `httpx`, the body looks like:

```html
<!DOCTYPE html>
<html>
  <body>
    <div id="root"></div>
    <script src="/app.js"></script>
  </body>
</html>
```

Where's the content? **It isn't in the response.** It will only appear after the JavaScript runs. `httpx` doesn't run JavaScript. Neither does BeautifulSoup. You'll need a real browser engine (lecture 05: Playwright) — *or* you find the JSON API the JS is calling and hit that directly (lecture 07: APIs first).

### How do you tell which one you're looking at?

A simple test: **`view-source:` the page** in your browser, or `curl` it, and look for the content. If the headline you see in the rendered page is in the raw HTML, it's server-rendered. If you only see `<div id="root"></div>` and a `<script>` tag, it's client-rendered.


In [ ]:
# A quick programmatic version of the same test
import httpx

def is_probably_csr(url: str) -> bool:
    """Heuristic: tiny HTML body + a script tag often means client-rendered."""
    r = httpx.get(url, follow_redirects=True, timeout=10.0)
    body_size = len(r.text)
    has_root_div = "id=\"root\"" in r.text or "id=\"app\"" in r.text
    has_script = "<script" in r.text
    return body_size < 5000 and has_root_div and has_script

print("example.com:", is_probably_csr("https://example.com"))
# Try a CSR site if you have one in mind. Many SPAs (single-page apps) light up here.


## 8. DevTools: your most important tool

Open your browser. Press `F12` (or `Cmd+Opt+I` on Mac). The panel that opens is **DevTools**. You will live in here while writing crawlers. Three tabs you must learn:

1. **Elements / Inspector** — shows the live DOM tree. Right-click any element on the page → "Inspect" jumps you to its node. This is where you find CSS selectors for the data you want.
2. **Network** — shows every HTTP request the page made. Filter by `XHR` or `Fetch` to see the AJAX calls. *This is how you find hidden APIs* (lecture 07).
3. **Console** — a JavaScript REPL running in the page's context. Useful for trying out selectors with `document.querySelector(...)`.

### Mini-exercise

1. Open `https://news.ycombinator.com/` in your browser.
2. Open DevTools → Elements.
3. Right-click the first headline → Inspect.
4. Notice that the headline text is *right there* in the HTML — server-rendered.
5. Now open `https://twitter.com/explore` in a new tab.
6. Right-click a tweet → Inspect.
7. Notice the much messier, deeply-nested div soup. Open the Network tab, refresh, filter by `Fetch/XHR`. You'll see JSON requests carrying the actual tweet data. Client-rendered.

Keep this distinction alive in your head. We'll come back to it.


## 9. Recap

- The web is HTTP request → HTTP response. Everything else is layered on top.
- Every HTTP request has a method, a path, headers, optionally a body. Every response has a status code, headers, and a body.
- HTML is a tree (the DOM). CSS styles it; JavaScript modifies it.
- Pages are either **server-rendered** (HTML has the data) or **client-rendered** (HTML is empty, JS fills it in). Your strategy depends on which.
- DevTools is non-negotiable. Get comfortable with Elements, Network, and Console.

## Exercises

1. Pick three websites you visit often. For each, view-source and decide: SSR or CSR? Write one sentence justifying your answer.
2. Use `httpx` to fetch each one. Compare the body length to what you see in the browser. Does it match your prediction from exercise 1?
3. Pick a site with a search box. Type a query, press Enter, watch the Network tab. Did the page do a full reload (`Doc` request) or fire an XHR? What does that tell you about its rendering strategy?

## Up next

In **Lecture 01** we'll talk about the *non-technical* foundations: what crawling is in the broader sense, the difference between crawling and scraping, and the rules — written and unwritten — that you should follow when pointing your code at someone else's server.
